# Этап 1: реранкинг и отказ на готовом OSNet

Изолированная проверка 13 порядков и 6 оценок уверенности. Без обучения, CLIP и изменения MVP.
Все настройки выбираются на calibration, фиксируются, затем проверяются на validation и ручных масках.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'training/rerank_score_control.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри Car-classification-MSK')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training.rerank_score_control import run, EXPERIMENT
OUTPUT = EXPERIMENT / 'results/run_01'
print('Python:', sys.executable)
print('Output:', OUTPUT)


## Один запуск с заморозкой выбора

Не меняйте коэффициенты по validation. После завершения повторный Run All проверяет hashes и читает отчёт.
При прерывании сохраняется выбор calibration. Не запускайте два kernel в одну выходную папку.
Основной этап использует готовые эмбеддинги; MPS и обучение не нужны.


In [ ]:
report = run(OUTPUT)


In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'RESULTS.md').read_text(encoding='utf-8')))


In [ ]:
import json
import pandas as pd
calibration = json.loads((OUTPUT / 'calibration.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'variant': x['config']['name'], **x['metrics_at_baseline_threshold']} for x in calibration['leaderboard']]))


## Что сравнивать

baseline → ranking_only показывает вклад порядка; baseline → refusal_only — вклад уверенности и порога.
combined объединяет оба изменения. Не выбирать вариант повторно по этой таблице validation.
Текущий MVP, исходные веса, изображения и index сохранены. Локальные CSV лежат в OUTPUT/validation, это не официальный test.
